# exp081 inference: 10s sliding window (Tucker mel)

**Setup**:
- Window: 10s
- Mel: Tucker spec (n_mels=256, n_fft=2048, hop=512)
- Sliding step: 5s → 12 windows/file = 12 row outputs
- Pad audio: 2.5s + 60s + 2.5s = 65s
- Smoothing: [0.1, 0.2, 0.4, 0.2, 0.1] across 12 windows
- Inference: PyTorch CPU (eca_nfnet_l0、Kaggle 90min limit)

**Input**:
- Kaggle Dataset: `maekeso/birdclef2026-exp081-weights` (.pth from exp081 R2 training)
- Competition: `birdclef-2026/test_soundscapes/`

**Output**: `/kaggle/working/submission.csv`


In [ ]:
# timm is pre-installed in Kaggle competition environment (internet=off)
import sys, os, time, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import librosa
import timm
import tqdm.auto as tqdm
from scipy.ndimage import convolve1d  # ★ exp017 と同じ discrete kernel smoothing

DEVICE = torch.device("cpu")  # Kaggle submission = CPU only
torch.set_num_threads(4)
print(f"Python: {sys.version[:50]}, torch: {torch.__version__}, timm: {timm.__version__}")
START = time.time()


In [ ]:
# CFG (must match training)
SR = 32_000
WINDOW_SEC = 10
N_WINDOWS_OUT = 12   # output rows per file
STEP_SEC = 5         # sliding step
N_CLASSES = 234

# Mel (Tucker spec)
N_MELS = 256
N_FFT = 2048
HOP_LENGTH = 512
F_MIN = 20
F_MAX = 16000

# Padded total length for 12 windows: (12-1)*5 + 10 = 65s
PADDED_SEC = (N_WINDOWS_OUT - 1) * STEP_SEC + WINDOW_SEC   # 65
PAD_LEAD_SEC = (PADDED_SEC - 60) / 2                       # 2.5s
PAD_TRAIL_SEC = PADDED_SEC - 60 - PAD_LEAD_SEC             # 2.5s
PADDED_SAMPLES = SR * PADDED_SEC
PAD_LEAD_SAMPLES = int(SR * PAD_LEAD_SEC)
PAD_TRAIL_SAMPLES = int(SR * PAD_TRAIL_SEC)
WINDOW_SAMPLES = SR * WINDOW_SEC
STEP_SAMPLES = SR * STEP_SEC

print(f"WINDOW: {WINDOW_SEC}s, STEP: {STEP_SEC}s")
print(f"PADDED: {PADDED_SEC}s ({PAD_LEAD_SEC}s lead + 60s + {PAD_TRAIL_SEC}s trail)")
print(f"N_WINDOWS_OUT: {N_WINDOWS_OUT}")

def find_dir(candidates):
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return None

DATA_PATH = find_dir([
    "/kaggle/input/competitions/birdclef-2026",
    "/kaggle/input/birdclef-2026",
])
assert DATA_PATH is not None
TEST_SC_DIR = Path(DATA_PATH) / "test_soundscapes"
SAMPLE_SUB = Path(DATA_PATH) / "sample_submission.csv"

PTH_DIR = find_dir([
    "/kaggle/input/birdclef2026-exp081-weights",
    "/kaggle/input/datasets/maekeso/birdclef2026-exp081-weights",
])
assert PTH_DIR is not None, "exp081 weights not mounted"

# ★ R2 専用: r2_* ckpt を最優先
PTH_PATH = None
for cand_name in ["r2_ckpt_best_ns22.pth", "r2_ckpt_best_macro.pth", "r2_ckpt_latest.pth",
                  "ckpt_best_ns22.pth", "ckpt_best_macro.pth"]:
    for cand in PTH_DIR.rglob(cand_name):
        PTH_PATH = cand
        break
    if PTH_PATH is not None:
        break
assert PTH_PATH is not None, f"No ckpt found in {PTH_DIR}"
print(f"Loading: {PTH_PATH} ({PTH_PATH.stat().st_size/1e6:.1f} MB)")

OUT_DIR = Path("/kaggle/working")


In [ ]:
# Model architecture (must match training: BirdSEDModel)
class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))


class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name, num_classes=N_CLASSES, drop_path_rate=0.0, hidden_dim=512,
                 use_perch_distill=True):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = WINDOW_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        if use_perch_distill:
            self.distill_head = DistillHead(self.backbone_dim, 1536)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        h_cls = self.gem_freq(h)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise, dim=2)
        if return_framewise:
            return clip_logits, framewise.permute(0, 2, 1)  # (B, time, num_class)
        return clip_logits


def sigmoid_np(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)


def load_birdsed_model(ckpt_path, backbone_name):
    model = BirdSEDModel(backbone_name=backbone_name).to(DEVICE)
    ckpt = torch.load(str(ckpt_path), weights_only=False, map_location="cpu")
    state = ckpt.get("model_state", ckpt.get("state_dict", ckpt))
    msg = model.load_state_dict(state, strict=False)
    model.eval()
    print(f"  {backbone_name}: missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}")
    if len(msg.missing_keys) > 5:
        print(f"    sample missing: {list(msg.missing_keys)[:3]}")
    if len(msg.unexpected_keys) > 5:
        print(f"    sample unexpected: {list(msg.unexpected_keys)[:3]}")
    return model


In [ ]:
# Load ckpt
ckpt = torch.load(str(PTH_PATH), weights_only=False, map_location="cpu")
print(f"  ckpt keys: {list(ckpt.keys())[:10]}")
print(f"  epoch: {ckpt.get('epoch', '?')}")
print(f"  val_ns22: {ckpt.get('val_ns22', ckpt.get('best_ns22', '?'))}")
print(f"  val_macro: {ckpt.get('val_macro', ckpt.get('best_macro', '?'))}")

BACKBONE = "eca_nfnet_l0"
model = BirdSEDModel(backbone_name=BACKBONE).to(DEVICE)

state = ckpt.get("model_state", ckpt.get("state_dict", ckpt))
msg = model.load_state_dict(state, strict=False)
model.eval()
print(f"  Load: missing={len(msg.missing_keys)}, unexpected={len(msg.unexpected_keys)}")
print(f"  Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


In [ ]:
# Mel transform (must match training)
mel_transform = T.MelSpectrogram(
    sample_rate=SR, normalized=True, n_fft=N_FFT,
    hop_length=HOP_LENGTH, win_length=N_FFT,
    f_max=F_MAX, n_mels=N_MELS, f_min=F_MIN,
).to(DEVICE)
amp_to_db = T.AmplitudeToDB(top_db=80).to(DEVICE)

def compute_mel(wav_tensor):
    """wav_tensor: (B, 1, n_samples). Returns mel (B, 1, n_mels, n_frames)."""
    mel = mel_transform(wav_tensor)  # (B, 1, n_mels, n_frames)
    mel = amp_to_db(mel)
    # Per-sample normalize
    B = mel.size(0)
    for i in range(B):
        mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
    return mel


# Sample submission for column order
sample_sub = pd.read_csv(SAMPLE_SUB)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
assert len(PRIMARY_LABELS) == N_CLASSES
print(f"Primary labels: {len(PRIMARY_LABELS)}")


In [ ]:
# Sliding inference function (★ exp017 paradigm: LOGIT-space blend + smoothing, sigmoid LAST)
GAUSSIAN_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1], dtype=np.float32)

def infer_file(audio_60s):
    """audio_60s: np.ndarray of shape (60s × SR,) = (1920000,).
    Returns: predictions (N_WINDOWS_OUT=12, N_CLASSES=234) as probability.

    Pipeline (Tucker/Babych/exp017 standard):
    1. Pad audio: 2.5s lead + 60s + 2.5s trail = 65s
    2. Slice 12 sliding windows (10s each, step 5s)
    3. Mel → model → clip_logits + frame_max
    4. Blend in LOGIT space: 0.5*clip + 0.5*frame_max
    5. Smooth LOGITS with [0.1, 0.2, 0.4, 0.2, 0.1] kernel (NOT prob space!)
    6. Sigmoid AT THE END
    """
    # 1. Pad: 2.5s lead + 60s + 2.5s trail = 65s
    audio_padded = np.zeros(PADDED_SAMPLES, dtype=np.float32)
    audio_padded[PAD_LEAD_SAMPLES:PAD_LEAD_SAMPLES + len(audio_60s)] = audio_60s[:60 * SR]

    # 2. Slice into 12 windows
    chunks = np.zeros((N_WINDOWS_OUT, WINDOW_SAMPLES), dtype=np.float32)
    for k in range(N_WINDOWS_OUT):
        start = k * STEP_SAMPLES
        end = start + WINDOW_SAMPLES
        chunks[k] = audio_padded[start:end]

    # 3. To tensor (B=12, 1, n_samples)
    wav_t = torch.from_numpy(chunks).unsqueeze(1).to(DEVICE)
    mel = compute_mel(wav_t)

    # 4. Model forward — keep as LOGITS, blend in logit space
    with torch.no_grad():
        clip_logit, framewise = model(mel, return_framewise=True)
        frame_max = framewise.max(dim=1).values
        blend_logits = 0.5 * clip_logit + 0.5 * frame_max
        blend_logits_np = blend_logits.float().cpu().numpy()  # (12, 234)

    # 5. Smoothing in LOGIT space (Tucker/Babych/exp017 standard)
    logits_smoothed = convolve1d(blend_logits_np, GAUSSIAN_KERNEL, axis=0,
                                 mode="nearest").astype(np.float32)

    # 6. Sigmoid LAST
    probs = sigmoid_np(logits_smoothed)
    return probs


In [ ]:
# Inference loop on test_soundscapes
test_files = sorted(TEST_SC_DIR.glob("*.ogg"))
print(f"Test files: {len(test_files)}")

rows = []
t0 = time.time()
for fi, fp in enumerate(tqdm.tqdm(test_files, desc="Infer")):
    try:
        y, _ = librosa.load(str(fp), sr=SR, mono=True)
    except Exception as e:
        print(f"  load fail {fp.name}: {e}")
        y = np.zeros(SR * 60, dtype=np.float32)

    target = SR * 60
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]

    probs = infer_file(y)  # (12, 234)

    file_stem = fp.stem
    for k in range(N_WINDOWS_OUT):
        end_sec = (k + 1) * 5
        row_id = f"{file_stem}_{end_sec}"
        rows.append([row_id] + probs[k].tolist())

    if (fi + 1) % 50 == 0 or fi == len(test_files) - 1:
        elapsed = time.time() - t0
        rate = (fi + 1) / max(elapsed, 0.001)
        eta = (len(test_files) - fi - 1) / max(rate, 0.001) / 60
        print(f"  [{fi+1}/{len(test_files)}] {elapsed:.0f}s rate={rate:.2f}f/s eta={eta:.1f}min")

print(f"\nInference DONE: {(time.time()-t0)/60:.1f} min, {len(rows)} rows")


In [ ]:
# Build submission CSV
sub_df = pd.DataFrame(rows, columns=["row_id"] + PRIMARY_LABELS)
print(f"sub_df: {sub_df.shape}")

# Sanity
assert sub_df["row_id"].nunique() == len(sub_df), "Duplicate row_id"
print(f"  mean prob: {sub_df[PRIMARY_LABELS].mean().mean():.5f}")
print(f"  max prob:  {sub_df[PRIMARY_LABELS].max().max():.4f}")
print(f"  NaN: {sub_df[PRIMARY_LABELS].isna().sum().sum()}")

# Reindex to match sample_submission order
if len(test_files) > 0 and len(sample_sub) > 0:
    sub_df = sub_df.set_index("row_id").reindex(sample_sub["row_id"]).reset_index()
    sub_df[PRIMARY_LABELS] = sub_df[PRIMARY_LABELS].fillna(0.0)

sub_df.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'submission.csv'} ({(OUT_DIR / 'submission.csv').stat().st_size/1e6:.1f} MB)")
print(f"Total time: {(time.time()-START)/60:.1f} min")
